# PESO SCAN — genuine-note denomination training

This notebook trains a six-class MobileNetV3 Small image classifier for ₱20, ₱50, ₱100, ₱200, ₱500, and ₱1,000. It does not detect counterfeits or authenticate notes. It contains no dataset or model weights.

Before starting, upload/copy the complete `ai-training` folder to `MyDrive/PESO_SCAN/ai-training`. Put your private photographs in `data/raw/` and your completed `data/annotations.csv` beside the `scripts/` folder. Do not share the Drive folder or notebook with people who should not see the photographs.

Colab's free GPU, memory, runtime length, and availability vary. The run can be repeated after a disconnect; keep the best saved checkpoint.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, time
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/PESO_SCAN/ai-training')
assert (DRIVE_PROJECT / 'scripts' / 'train.py').is_file(), f'Project scripts not found under {DRIVE_PROJECT}'
assert (DRIVE_PROJECT / 'data' / 'annotations.csv').is_file(), 'Add data/annotations.csv first'
assert (DRIVE_PROJECT / 'data' / 'raw').is_dir(), 'Add photographs under data/raw/'
WORK = Path('/content/peso-scan/ai-training')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(DRIVE_PROJECT, WORK, ignore=shutil.ignore_patterns('__pycache__', '.git'))
print('Runtime copy ready:', WORK)

## Runtime and settings

The notebook leaves Colab's preinstalled, compatible `torch` and `torchvision` versions intact and installs only the additional export/runtime packages. `PRETRAINED = False` avoids using external ImageNet weights by default. Read `LICENSES.md` before changing that setting or shipping artifacts commercially.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt'), '-r', str(WORK / 'requirements-colab.txt')])
import torch, torchvision
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (GPU not assigned)')
EPOCHS = 20
BATCH_SIZE = 16
SEED = 42
PRETRAINED = False  # Set True only after reviewing ImageNet checkpoint terms in LICENSES.md
RUN_ID = time.strftime('%Y%m%d-%H%M%S')
OUTPUT = WORK / 'runs' / RUN_ID
print('Output:', OUTPUT)

## Validate and create leakage-safe splits

This checks the photographs and annotation metadata, then assigns every view of each physical note to a single split. At least three distinct physical notes per denomination are required just to populate train/validation/test; that is far too little for a trustworthy model.

In [ ]:
subprocess.check_call([sys.executable, str(WORK / 'scripts' / 'validate_dataset.py'), '--annotations', str(WORK / 'data' / 'annotations.csv'), '--images-dir', str(WORK / 'data' / 'raw')])
subprocess.check_call([sys.executable, str(WORK / 'scripts' / 'prepare_dataset.py'), '--annotations', str(WORK / 'data' / 'annotations.csv'), '--images-dir', str(WORK / 'data' / 'raw'), '--output-dir', str(WORK / 'data' / 'processed'), '--seed', str(SEED)])
MANIFEST = WORK / 'data' / 'processed' / 'split_manifest.csv'
IMAGES = WORK / 'data' / 'raw'

## Train

Only training images receive mild augmentation. Validation selects the saved checkpoint. The test split is not used for checkpoint selection. Training can run on CPU if Colab did not assign a GPU, but will be slower.

In [ ]:
command = [sys.executable, str(WORK / 'scripts' / 'train.py'), '--manifest', str(MANIFEST), '--images-dir', str(IMAGES), '--output-dir', str(OUTPUT), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED)]
if PRETRAINED: command.append('--pretrained')
subprocess.check_call(command)
CHECKPOINT = OUTPUT / 'best_model.pt'
assert CHECKPOINT.is_file(), 'Training did not produce a real checkpoint'

## Evaluate held-out test notes

Run this evaluation only after training decisions are final. Review per-class precision/recall and the confusion matrix as well as aggregate accuracy. Metrics from a small or narrowly collected dataset do not establish real-world performance or authenticity.

In [ ]:
subprocess.check_call([sys.executable, str(WORK / 'scripts' / 'evaluate.py'), '--manifest', str(MANIFEST), '--images-dir', str(IMAGES), '--checkpoint', str(CHECKPOINT), '--output-dir', str(OUTPUT)])
print((OUTPUT / 'test_metrics.json').read_text())

## Export ONNX and save artifacts to Drive

Export runs a numerical parity check against ONNX Runtime on random tensors. That checks the conversion path only; it does not test model quality. The random tensors are not banknote images or training data.

In [ ]:
subprocess.check_call([sys.executable, str(WORK / 'scripts' / 'export_onnx.py'), '--checkpoint', str(CHECKPOINT), '--output', str(OUTPUT / 'peso_denomination.onnx'), '--metadata', str(OUTPUT / 'model_metadata.json')])
DRIVE_ARTIFACTS = DRIVE_PROJECT / 'artifacts' / RUN_ID
DRIVE_ARTIFACTS.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT, DRIVE_ARTIFACTS)
print('Saved real training artifacts to:', DRIVE_ARTIFACTS)
print('No artifact was produced until the training and export cells completed successfully.')